# Week 7 · DPO 偏好对齐

> **本周一句话**:用规则评分器自动标偏好对(chosen vs rejected),用 DPO loss 把 v0.9 升级到 v0.10 —— 不训独立 reward model,直接让 policy 学会偏好。

这周难点是**理解 DPO 的数学**(为什么这个 loss 等价于 RLHF)和**正确的 ref_model 用法**(eval / train 模式切换是常见 bug)。

## 0. 本周目标

| 阶段 | 数据 | 方法 | 产出 |
|---|---|---|---|
| 评分器 | — | pypinyin 提韵尾、句数/字数判定 | `evaluation/score.py` |
| 数据构造 | SFT val 中 1000 个 prompt × 4 候选 | 排序选最高最低分,过滤分差 < 5 | ~500 对 (chosen, rejected) |
| DPO 训练 | 上述偏好对 | 500 步,lr=1e-5,β=0.1 | v0.10 |
| 评估 | 6 题 head-to-head | 同种子比 v0.9 vs v0.10 | win/lose/tie |

**val loss 概念变化**:不再是"语言模型 loss",而是 DPO loss = `-log σ(β·margin)`。理想:< 0.69(= -log(0.5),代表模型至少能 50% 正确偏好 chosen)。

## 1. 前置知识

**必备**:
- Week 6 SFT 跑通,v0.9 能按题目+风格生成
- 知道 RLHF 是什么(reward model + PPO 的传统流程)

**这周第一次遇到**:
- `pypinyin` 库:从汉字提取韵母
- DPO loss 公式与推导
- "policy model" vs "reference model" 的概念
- 训练时 train / eval 模式切换的微妙之处

## 2. 核心概念

### 2.1 RLHF 三步走 vs DPO 一步到位

传统 RLHF(InstructGPT / ChatGPT 走的):

```
SFT → 训 Reward Model → 用 PPO 优化 policy(以 RM 为奖励信号)
```

问题:
- 需要训独立 RM(再多一个模型 + 数据)
- PPO 调超参很难,训练不稳定

DPO(Direct Preference Optimization,2023)证明:**给定偏好对 (chosen, rejected),可以直接优化 policy 而不显式训 RM**,数学上等价于 RM + KL 约束的最优解。

一行 loss 公式:

$$L_{\mathrm{DPO}} = -\log \sigma\left(\beta \cdot \left(\log\frac{\pi(c)}{\pi_{\mathrm{ref}}(c)} - \log\frac{\pi(r)}{\pi_{\mathrm{ref}}(r)}\right)\right)$$

直觉:让当前 policy 在 chosen 上的相对 log-prob(相对于 ref)比 rejected 高。**β 越大,模型越激进地偏好 chosen,但越容易偏离 ref(灾难性遗忘)**。我们用 β=0.1。

### 2.2 评分器是 DPO 的"老师"

没有人工标注,我们用规则评分器:

```python
def score_poem(poem, target_style, target_title):
    score = 0
    # 1. 句数 4/8 对 → +10
    # 2. 每句字数 5/7 对 → 每句 +2
    # 3. 偶数句尾字押韵(pypinyin 提韵) → 每对 +4
    # 4. 题目相关字出现 → 每个 +3
    # 5. 无连续 3 次以上重复 → +3
    return score
```

**评分器的智力上限就是 DPO 的上限**。我们的规则评分能识别格律 + 押韵 + 字面相关,但识别不了意境、情感、用典是否准确。所以 v0.10 在"形式"上会比 v0.9 好,在"意境"上未必。

GPT-4-as-a-judge 是更强的自动评分,但本课程用规则版,可解释、可复现。

### 2.3 偏好对的构造

```
for prompt in 1000 个 prompts:
    candidates = [SFT model 生成 N=4 次]   # T=1.0 + top_k=40,刻意高温度求多样性
    打分排序
    chosen   = 最高分
    rejected = 最低分
    if chosen.score - rejected.score >= 5:  # 必须有显著分差,否则跳过
        save (chosen, rejected)
```

**关键设计**:
- 温度 1.0 + top_k 40:刻意求多样性,4 个候选差别要大才能挑出"明显好"和"明显差"
- 分差阈值 5:避免"两首差不多的诗"训成偏好对(噪声大,反而有害)
- 用 SFT val 而不是 train 的 prompt:确保 prompt 不在 v0.9 训练分布里

最终大约能收 500-700 对(取决于 SFT 模型质量和评分分布)。

### 2.4 ref_model 的角色

DPO 需要两个模型:
- **policy**(可训): 一开始是 v0.9 SFT,训练中被更新
- **ref**(冻结): 一直是 v0.9 SFT 的快照,提供"原始概率"作为对比基准

代码:

```python
model     = load_v09_sft()              # 待训
ref_model = copy.deepcopy(model)         # 冻结
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad = False
```

**为什么需要 ref**?
- 直接最大化 log π(chosen) 会让模型走极端(永远只生成 chosen 那种格式)→ 失去多样性、灾难性遗忘
- 用 `log π / π_ref` 做相对量,**相对** ref 的偏好提升 = 学到的是"偏好方向"而不是"绝对概率"
- 数学上等价于"在 RM 优化目标里加一个 KL(π || π_ref) 约束",防止 policy 跑得太远

### 2.5 train / eval 模式的关键陷阱

**最容易踩的坑**:dropout 开/关导致 ref 和 policy 算出不同 log-prob。

```python
# 训练循环里
model.train()        # ← policy 开 Dropout (正常训练状态)
loss, m = dpo_loss(model, ref_model, ...)

# 评估循环里
model.eval()         # ← policy 必须 eval,关 Dropout
ref_model.eval()     # ← ref 永远 eval
loss, m = dpo_loss(model, ref_model, ...)
```

**为什么训练时 policy 可以开 Dropout**?Dropout 引入正则,让 policy 不要 overfit 偏好对。但 ref 必须冻结(eval),否则 `model_sft.eval()` 调用同时影响 policy 和 ref(同一个对象的克隆,deepcopy 之后是独立的,但容易写错)。

**初学者常犯**:整个训练循环 `model.train()` 一开到底,评估时也是 train 模式 → ref 和 policy 都开 Dropout → DPO loss 计算不一致。

## 3. 代码地图

| 文件 | 行数 | 干什么 |
|---|---|---|
| `evaluation/score.py` | 90 | pypinyin 提韵 + 评分函数 |
| `data/prepare_dpo.py` | 110 | 用 v0.9 生成 + 评分 → 偏好对 |
| `train/dpo_loss.py` | 70 | `compute_logprobs` + `dpo_loss`,带 metrics |
| `train/train_v10_dpo.py` | 175 | 完整训练循环,500 步 |
| `inference/compare_dpo.py` | 60 | 同种子对比 v0.9 vs v0.10 |

## 4. 动手做

In [ ]:
import subprocess, sys


def run(cmd):
    """跑子进程并把输出实时打印到 cell。
    subprocess.run 默认把子进程 stdout 写到 kernel 原始 fd, notebook 看不到;
    用 Popen 逐行回读才能在 cell 里实时看到脚本的 print。
    "python" 换成 sys.executable, 确保用当前 kernel 解释器。"""
    cmd = [sys.executable if c == "python" else c for c in cmd]
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        raise SystemExit(f"{cmd} 退出码 {proc.returncode}")

# 1. 生成偏好对(~30 分钟 T4,大头是 SFT 模型推理)
run(["python", "../data/prepare_dpo.py"])
# 预期:
#   "采样 1000 个唯一 prompt"
#   每 100 个 print 一次进度
#   最终收 500-700 对,平均分差 8-12

In [ ]:
# 2. DPO 训练(~5 分钟)
run(["python", "../train/train_v10_dpo.py"])
# 预期:
#   step 0 时 val loss ~0.69(随机猜对偏好的水平)
#   收敛到 < 0.5,acc > 0.7

In [ ]:
# 3. 同种子对比 v0.9 vs v0.10
run(["python", "../inference/compare_dpo.py"])
# 预期: v10 在"形式合规"维度上 win 多 lose 少(评分器只衡量形式)
#       意境层面 v9 / v10 可能各有千秋

## 5. 自测题

**A. DPO 数学**
- A1 推导:当 ref == policy 时,DPO loss 严格等于多少?为什么?
- A2 β 从 0.1 改成 1.0,训练会有什么不同?极端情况 β = 10 呢?
- A3 DPO loss 收敛到 0 意味着什么?健康吗?

**B. 偏好对**
- B1 评分器给 chosen 50 分、rejected 48 分,这样的对该不该用?为什么?
- B2 N_CANDIDATES=4 改成 8 会怎样?改成 2 呢?
- B3 评分器只看"形式"(押韵 / 句数),DPO 训完模型只能在形式上"更好"吗?能学到意境吗?

**C. ref_model**
- C1 为什么不能省略 ref_model,直接最大化 log π(chosen)?
- C2 ref 用 v0.8(预训练)而不是 v0.9(SFT)会怎样?
- C3 训练过程中能不能定期把 policy 复制成新的 ref?(这叫 iterative DPO)

**D. 工程**
- D1 评估时 `model_sft.eval()` 但忘了切回 train,接下来训练循环会怎样?
- D2 训 500 步够用吗?训 5000 步会怎样?
- D3 DPO weight_decay=0 不加权重衰减,原因?

## 6. 容易踩的坑

**坑 1:Dropout 开着评估 → DPO loss 不是 0.6931 (理论值)**

在 sanity check 阶段(model_state == ref_state 时),如果不切 eval,Dropout 随机化让两边 logits 不一样,loss 会偏离 0.6931。**我们的代码在 estimate_dpo 开头显式 `model.eval()`**。

**坑 2:`compute_logprobs` 对 padding 算了 loss**

答案末尾 + pad 全要 mask 掉。我们的实现做了 `targets != pad_id` 的 mask。否则 pad 越多 loss 越虚高。

**坑 3:偏好对的 prompt_lens 算错**

compute_logprobs 里 mask 起点是 `prompt_lens - 1`(因为 per_token_logp 是错位的)。少 1 多 1 都会让 loss 算到 prompt 上(或漏算第一个答案 token)。

**坑 4:训练 200 步后突然 grad_norm 飙到 30+**

DPO 训练遇到几个"分差极大"的对时,梯度会突然变大。我们有 `clip_grad_norm_(1.0)` 兜底。但如果整体平均 grad_norm 都在涨,说明 β 太大或 lr 太大,要降。

**坑 5:reward_chosen 涨,reward_rejected 没降 —— 模型学了"绝对偏好"而不是"相对偏好"**

DPO 的健康曲线是:reward_chosen 涨,reward_rejected 跌,两者差 (margin) 越拉越大。如果只是 chosen 涨而 rejected 没动,说明 β 太小 / KL 约束太强,模型只往 chosen 拉但不敢拒绝 rejected。

## 7. 进入 Week 8 前

现在你应该:
- ☑ `dpo_data.pt` 存在,500-700 对偏好
- ☑ `checkpoints/v10_dpo/best.pt` 存在
- ☑ compare_dpo 跑出来 v10 评分 win > lose
- ☑ 能在白板上推 DPO loss 公式(至少 ref==policy → 0.6931 这一步)

Week 8 完全换赛道 —— 不再训我们自己的小模型,而是 4-bit 量化加载 Qwen-1.5B + LoRA 微调。**对照自己写的 25M 模型,看商业级模型差在哪**。